# ZS-BERT — m=10 — one independent run
Select dataset and RUN_ID (1–5 → seeds 300–304). Save Version → Save & Run All.
GPU + Internet required. Data uses versioned BTL snapshots with SHA256; no dataset Input needed. Author Drive identity is unverified.
A paused run is incomplete: attach its output and resume the same seed.
Official run: 10 epochs follow authors' README; best selects on test F1, final reports epoch 10.
Only complete runs count. If paused, resume the same run; 30 runs can require more than 30 sessions.
Wiki-ZSL uses the project snapshot of 93,383 samples, not the paper-reported 94,383.
This implementation follows the existing repo model/loss and records known reproduction limits.


In [1]:
import time
SESSION_STARTED = time.time()
DATASET = "fewrel"  # "fewrel" or "wikizsl"
RUN_ID = 1
M = 10
CODE_REV = "b4b84208cf43b4274f2d2aaaf14beead0daf4707"  # After first validation, replace with the exact tested commit SHA.
DATA_SOURCE = "snapshot"  # "snapshot" or "author-drive"; project snapshot avoids Google Drive errors.
DATA_FILE = ""  # Optional local/input override; default uses the project snapshot.
EXPECTED_DATA_SHA256 = {
    "fewrel": "85c2595bf743c203c22e78868f5a9f309ad726aa27c36e804554d99461736e19",
    "wikizsl": "9a16f9752d69152e32d5ac0b59d48095f7e530d8ccc244f1e48cd48110d84e47",
}[DATASET]
ALLOW_WIKI_COUNT_MISMATCH = DATASET == "wikizsl"  # Project Wiki snapshot: 93,383 samples; paper: 94,383.
RESUME_DIR = ""  # Trusted previous output directory, only if continuing this same run.
MAX_SESSION_HOURS = 10.5
SMOKE_TEST = False  # Official run: complete all 10 epochs.


## Setup
No access token is required for the public GitHub repository. The actual commit is printed and saved.

In [2]:
import os, sys, subprocess
from pathlib import Path
os.environ.update(PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false",
    CUBLAS_WORKSPACE_CONFIG=":4096:8", HF_HUB_DISABLE_XET="1", USE_TF="0", USE_FLAX="0",
    PYTHONHASHSEED="0")
WORK = Path("/kaggle/working")
REPO = WORK / "zs-bert-simulation"
URL = "https://github.com/Student-persist-1919/zs-bert-simulation.git"
if not REPO.exists():
    subprocess.run(["git", "clone", URL, str(REPO)], check=True)
if subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip() != URL:
    raise RuntimeError("Unexpected repository origin")
# Fetch advertised refs, then resolve branch/tag/full or abbreviated commit locally.
# GitHub does not treat a short SHA as a remote ref for `git fetch origin SHORT`.
subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
resolved = None
for revision_ref in [f"refs/remotes/origin/{CODE_REV}", CODE_REV]:
    try:
        resolved = subprocess.check_output(
            ["git", "-C", str(REPO), "rev-parse", "--verify", f"{revision_ref}^{{commit}}"],
            text=True, stderr=subprocess.DEVNULL).strip()
        break
    except subprocess.CalledProcessError:
        continue
if resolved is None:
    raise RuntimeError(f"Cannot resolve CODE_REV={CODE_REV!r}; use a branch, tag or full commit SHA present in repository history.")
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", resolved], check=True)
print("CODE COMMIT:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True))
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO/"requirements-kaggle.txt")], check=True)
subprocess.run([sys.executable, "-c", "from transformers import BertModel; from sentence_transformers import SentenceTransformer; import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.cuda.get_device_name(0))"], check=True)


Cloning into '/kaggle/working/zs-bert-simulation'...
HEAD is now at b4b8420 Fix Kaggle Drive download failure with checksummed project dataset snapshots


CODE COMMIT: b4b84208cf43b4274f2d2aaaf14beead0daf4707

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 52.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 96.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 15.5 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.3
    Uninstalling tokenizers-0.23.3:
      Successfully uninstalled tokenizers-0.23.3
  Attempting uninstall: gdown
    Found existing installation: gdown 5.2.2
    Uninstalling gdown-5.2.2:
      Successfully uninstalled gdown-5.2.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.20.2
    Uninstalling transformers-5.20.2:
      Successfully uninstalled transformers-5.20.2
  Attempting uninstall: sent

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


2.11.0+cu130 Tesla T4


CompletedProcess(args=['/usr/bin/python3', '-c', 'from transformers import BertModel; from sentence_transformers import SentenceTransformer; import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.cuda.get_device_name(0))'], returncode=0)

## Run and export
The deadline includes setup/download time. The evidence ZIP excludes the multi-GB checkpoint. Smoke tests run one epoch on four samples per relation and cannot enter official results.

In [3]:
assert DATASET in ("fewrel", "wikizsl") and RUN_ID in range(1, 6)
assert 0 < MAX_SESSION_HOURS <= 11
cmd = [sys.executable, "-u", str(REPO/"scripts/kaggle_run.py"),
       "--dataset", DATASET, "--data-source", DATA_SOURCE, "--m", str(M), "--run-id", str(RUN_ID),
       "--deadline", str(SESSION_STARTED + MAX_SESSION_HOURS * 3600)]
for flag, value in [("--resume", RESUME_DIR), ("--data-file", DATA_FILE),
                    ("--expected-data-sha256", EXPECTED_DATA_SHA256)]:
    if value:
        cmd += [flag, value]
if ALLOW_WIKI_COUNT_MISMATCH:
    cmd += ["--allow-wiki-count-mismatch"]
if SMOKE_TEST:
    cmd += ["--smoke"]
subprocess.run(cmd, check=True)


DATA SOURCE: versioned project snapshot; identity with author Drive file has not been independently verified.
DATA AUDIT: {"dataset": "fewrel", "raw_sha256": "85c2595bf743c203c22e78868f5a9f309ad726aa27c36e804554d99461736e19", "samples": 56000, "relations": 80, "paper_samples": 56000}


Batches: 100%|██████████| 20/20 [00:01<00:00, 13.81it/s]


CONFIG: {"dataset": "fewrel", "m": 10, "epochs": 10, "batch_size": 4, "lr": 5e-06, "gamma": 7.5, "alpha": 0.4, "precision": "fp32", "dist_func": "inner", "eval_batch_size": 4, "gradient_checkpointing": true, "bert_revision": "06fa25dc1b6e034e91607fd67c247051ccaeff65", "sbert_revision": "a85b0d10e16867eb768e411c66f79475505b65dd", "author_revision": "e2ffaea66244b2905055ee7cc47917985c3eab60", "property_sha256": "d9c91ac01f18398e58a48ad7913e3f168553e3c06b306879b1b2389bde3eb21d", "data_sha256": "50a97fd62b947608192a32d5e8aa70dff5a0828f6d7f4f6b7b7fcb7981baca9c", "raw_data_sha256": "85c2595bf743c203c22e78868f5a9f309ad726aa27c36e804554d99461736e19", "attributes_sha256": "3268ca21d18508cb29ab66d75b7a49c51ff9f4c669d7aa294d3cf427923ce2dc", "descriptions_sha256": "b44661fabb07ef42225489cf9cb69f6cbbe38277fe3f4c48cc78004ae46ae1bf", "source_hashes": {"model/model.py": "8fc43e4aa96452d5856ab60bbca28f244e9396277e38cdc8b86bbd831ead9a12", "model/data_helper.py": "d53ccb25cf78e0a2c8e2e10fedf438c5887f6ae2

CompletedProcess(args=['/usr/bin/python3', '-u', '/kaggle/working/zs-bert-simulation/scripts/kaggle_run.py', '--dataset', 'fewrel', '--data-source', 'snapshot', '--m', '10', '--run-id', '1', '--deadline', '1795186061.8361468', '--expected-data-sha256', '85c2595bf743c203c22e78868f5a9f309ad726aa27c36e804554d99461736e19'], returncode=0)

## Preserve evidence
Download the `*_evidence.zip` from Output. Download the executed `.ipynb` from this exact Kaggle version.
For paused runs keep the full output directory (including checkpoint) on Kaggle; resume is the same run.
Locally import official evidence with:
`python scripts/collect_results.py --zip PATH --attempt original --kaggle-version-url VERSION_URL`
Store the executed notebook in `experiments/executed/` with dataset, m and seed in its name.
The collector regenerates `results.csv` and `RESULTS.md` beside these three notebooks.
